In [1]:
import os
# dagshub.init(repo_owner='basabelhu', repo_name='SimpleCompelteDataScienceProject', mlflow=True)
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/basabelhu/SimpleCompelteDataScienceProject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "basabelhu"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "ddb6cc955e7ecf9348fc890b9aa6f54cda7869ab"


In [2]:
pwd

'/Users/basazinbelhu/Simple_MLOPS/research'

In [4]:
# os.chdir("../")

In [6]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str 

In [8]:
from src.simple_mlops.constants import *
from src.simple_mlops.utils.common import read_yaml, create_directories, save_json

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifact_dir])
    
    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config = self.config.model_evaluation

        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        create_directories([config.root_dir])
        model_evaluation_config = ModelEvaluationConfig(root_dir = config.root_dir,
                                                        test_data_path = config.test_data_path,
                                                        model_path = config.model_path,
                                                        all_params = params, 
                                                        metric_file_name=config.metric_file_name, 
                                                        target_column=schema.name, 
                                                        mlflow_uri = "https://dagshub.com/basabelhu/SimpleCompelteDataScienceProject.mlflow")
        return model_evaluation_config

In [12]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [13]:
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from urllib.parse import urlparse
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.simple_mlops import logger
from src.simple_mlops.utils.common import save_json
# from src.simple_mlops.entity.config_entity import ModelEvaluationConfig


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            rmse, mae, r2 = self.eval_metrics(test_y, predicted_qualities)

            # save metrics locally as JSON
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            # log params and metrics to MLflow / DagsHub
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            # DagsHub does NOT support the model registry — only register on a real registry
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

        logger.info(f"Logged metrics to MLflow: {scores}")

In [14]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-06-13 10:17:13,548: INFO: common]: Successfully read YAML file: config/config.yaml
[2026-06-13 10:17:13,561: INFO: common]: Successfully read YAML file: params.yaml
[2026-06-13 10:17:13,565: INFO: common]: Successfully read YAML file: schema.yaml
[2026-06-13 10:17:13,567: INFO: common]: Directory created or already exists: artifact
[2026-06-13 10:17:13,570: INFO: common]: Directory created or already exists: artifact/model_evaluation
[2026-06-13 10:17:23,396: INFO: common]: Successfully saved JSON file: artifact/model_evaluation/metrics.json


2026/06/13 10:17:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 10:17:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticNetModel'.
2026/06/13 10:17:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.


🏃 View run tasteful-lamb-40 at: https://dagshub.com/basabelhu/SimpleCompelteDataScienceProject.mlflow/#/experiments/0/runs/2eeebae445fc4d6086bcd869e8166933
🧪 View experiment at: https://dagshub.com/basabelhu/SimpleCompelteDataScienceProject.mlflow/#/experiments/0
[2026-06-13 10:17:40,675: INFO: 1948616976]: Logged metrics to MLflow: {'rmse': np.float64(0.6846172586851057), 'mae': 0.5461216767438996, 'r2': 0.24231243705661587}
